<a href="https://colab.research.google.com/github/Gnoltd/BitcoinPredictionResearch-/blob/main/%5BCRYPTO%5D_TECHNICAL_INDICATORS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title # Mount Data
from google.colab import drive
import os

# Ensure the mount point is empty before mounting
if os.path.exists('/content/drive'):
    # Only remove if it's a directory and not already a mount point
    if os.path.isdir('/content/drive') and not os.path.ismount('/content/drive'):
        print("Clearing existing /content/drive directory...")
        for root, dirs, files in os.walk('/content/drive', topdown=False):
            for name in files:
                os.remove(os.path.join(root, name))
            for name in dirs:
                os.rmdir(os.path.join(root, name))
    elif os.path.isfile('/content/drive'):
        print("Removing file at /content/drive...")
        os.remove('/content/drive')

# Attempt to create the directory if it doesn't exist (it should be empty now)
os.makedirs('/content/drive', exist_ok=True)

drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
# @title #CRAWL DATA
import os
import pandas as pd
import numpy as np
import requests
import re
import warnings
from google.colab import drive

warnings.filterwarnings('ignore')

# Directory configuration
base_dir = '/content/drive/MyDrive/Crypto Research/DATA/Technical Indicators'
out_dir = os.path.join(base_dir, 'RawData')
os.makedirs(out_dir, exist_ok=True)
os.chdir(base_dir)

# Timeframe and target asset setup
start_date = '2023-01-01'
end_date = '2025-12-31'
coin = 'bitcoin'

# Define features to extract
raw_features_list = [
    'transactions', 'size', 'sentbyaddress', 'transactionfees',
    'blocktime', 'difficulty', 'hashrate', 'transactionvalue',
    'mediantransactionvalue', 'profitability', 'activeaddresses',
    'sentinusd', 'top100cap', 'fee-to-reward-ratio', 'mediantransactionfee'
]

# Include price for future log return calculation
features_to_fetch = raw_features_list + ['price']

# Data extraction function
def fetch_bitinfocharts_data(feature, coin):
    url = f"https://bitinfocharts.com/comparison/{coin}-{feature}.html#alltime"
    headers = {'User-Agent': 'Mozilla/5.0'}
    response = requests.get(url, headers=headers)

    pattern = r'\[new Date\("(.*?)"\),(.*?)\]'
    matches = re.findall(pattern, response.text)

    dates, values = [], []
    for match in matches:
        dates.append(pd.to_datetime(match[0]))
        val = match[1]
        values.append(float(val) if val != 'null' else np.nan)

    df = pd.DataFrame({'Date': dates, feature: values})
    df.set_index('Date', inplace=True)
    return df

# Execute data fetching loop
print("Fetching raw data...")
df_raw = pd.DataFrame()

for feature in features_to_fetch:
    try:
        df_temp = fetch_bitinfocharts_data(feature, coin)
        if df_raw.empty:
            df_raw = df_temp
        else:
            df_raw = df_raw.join(df_temp, how='outer')
        print(f"Successfully fetched: {feature}")
    except Exception as e:
        print(f"Error fetching {feature}: {e}")

# Save raw data to directory
output_filename = os.path.join(out_dir, f'{coin}_raw_data.csv')
df_raw.to_csv(output_filename)
print(f"Process completed. Raw data saved to: {output_filename}")

Fetching raw data...
Successfully fetched: transactions
Successfully fetched: size
Successfully fetched: sentbyaddress
Successfully fetched: transactionfees
Successfully fetched: blocktime
Successfully fetched: difficulty
Successfully fetched: hashrate
Successfully fetched: transactionvalue
Successfully fetched: mediantransactionvalue
Successfully fetched: profitability
Successfully fetched: activeaddresses
Successfully fetched: sentinusd
Successfully fetched: top100cap
Successfully fetched: fee-to-reward-ratio
Successfully fetched: mediantransactionfee
Successfully fetched: price
Process completed. Raw data saved to: /content/drive/MyDrive/Crypto Research/DATA/Technical Indicators/RawData/bitcoin_raw_data.csv


In [ ]:
# @title #SPLIT TRAIN/TEST
import pandas as pd
import numpy as np
import os
import warnings

warnings.filterwarnings('ignore')

# Configuration
start_date = '2023-01-01'
end_date = '2025-12-31'
coin = 'bitcoin'

base_dir = '/content/drive/MyDrive/Crypto Research/DATA/Technical Indicators'
raw_data_file = os.path.join(base_dir, 'RawData/bitcoin_raw_data.csv')
output_dir = os.path.join(base_dir, 'RawData')
os.makedirs(output_dir, exist_ok=True)

# Load raw data
print("Loading raw data...")
df_raw = pd.read_csv(raw_data_file, index_col=0, parse_dates=True)
full_date_range = pd.date_range(start=start_date, end=end_date, freq='D')
df_raw = df_raw.reindex(full_date_range)
df_raw.index.name = 'Date'

print(f"Raw data shape: {df_raw.shape}")
print(f"Date range: {df_raw.index.min()} to {df_raw.index.max()}")

# Define temporal boundaries
train_start = '2023-01-01'
train_end = '2025-05-31'
test_start = '2025-06-01'
test_end = '2025-12-31'

print(f"\nTemporal split configuration:")
print(f"  Train: {train_start} to {train_end}")
print(f"  Test:  {test_start} to {test_end}")

# Split train and test
print("\nSplitting data by date...")
df_train = df_raw.loc[train_start:train_end].copy()
df_test = df_raw.loc[test_start:test_end].copy()

print(f"Train shape: {df_train.shape}")
print(f"Test shape: {df_test.shape}")

# Verify no overlap
train_dates = set(df_train.index)
test_dates = set(df_test.index)
overlap = train_dates & test_dates

print(f"\nTemporal verification:")
print(f"  Train period: {df_train.index.min()} to {df_train.index.max()}")
print(f"  Test period: {df_test.index.min()} to {df_test.index.max()}")
print(f"  Overlapping dates: {len(overlap)}")

if len(overlap) == 0:
    print("  Status: No temporal overlap")
else:
    print("  Status: Warning - overlap detected")

# Export raw split data
train_file = os.path.join(output_dir, f'{coin}_train_raw.csv')
test_file = os.path.join(output_dir, f'{coin}_test_raw.csv')

df_train.to_csv(train_file)
df_test.to_csv(test_file)

print(f"\nRaw split data export:")
print(f"  Train: {train_file}")
print(f"  Test: {test_file}")

Loading raw data...


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Crypto Research/DATA/Technical Indicators/RawData/bitcoin_raw_data.csv'

In [ ]:
# @title # Pre-Processed Data (Train Only) + Applied to Test
import pandas as pd
import numpy as np
import os
import warnings
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
import pywt
import pickle

warnings.filterwarnings('ignore')

# Configuration and Paths
coin = 'bitcoin'
base_dir = '/content/drive/MyDrive/Crypto Research/DATA/Technical Indicators'
train_test_dir = os.path.join(base_dir, 'RawData')
processed_dir = os.path.join(base_dir, 'ProcessedData/TrainTest')
os.makedirs(processed_dir, exist_ok=True)

# Load Data
train_file = os.path.join(train_test_dir, f'{coin}_train_raw.csv')
test_file = os.path.join(train_test_dir, f'{coin}_test_raw.csv')

df_train = pd.read_csv(train_file, index_col='Date', parse_dates=True)
df_test = pd.read_csv(test_file, index_col='Date', parse_dates=True)

# Extract Raw Test Prices
df_raw_prices = df_test[['price']].copy()
df_raw_prices.rename(columns={'price': 'raw_close_price'}, inplace=True)

# Missing Values
train_missing = df_train.isna().stack()[lambda x: x].index.tolist()
if train_missing:
    print("Train Missing Values (Date, Column):")
    for date, col in train_missing:
        print(f"{date.date()} - {col}")
else:
    print("No missing values in Train Data.")

test_missing = df_test.isna().stack()[lambda x: x].index.tolist()
if test_missing:
    print("\nTest Missing Values (Date, Column):")
    for date, col in test_missing:
        print(f"{date.date()} - {col}")
else:
    print("\nNo missing values in Test Data.")

# Missing Data Imputation
  # Apply linear interpolation for smooth transition of continuous features
df_train.interpolate(method='linear', inplace=True)
df_test.interpolate(method='linear', inplace=True)

  # Apply fallback fills for any remaining NaNs at the exact boundaries
df_train.fillna(method='ffill', inplace=True)
df_train.fillna(method='bfill', inplace=True)
df_test.fillna(method='ffill', inplace=True)
df_test.fillna(method='bfill', inplace=True)

# Log Return Calculation
last_train_price = df_train['price'].iloc[-1]
df_train['log_return'] = np.log(df_train['price'] / df_train['price'].shift(1))

test_prices = pd.concat([pd.Series([last_train_price]), df_test['price']])
df_test['log_return'] = np.log(test_prices / test_prices.shift(1)).iloc[1:].values

# Clean Up Raw Price Columns
df_train.drop(columns=['price'], inplace=True)
df_test.drop(columns=['price'], inplace=True)
df_train.dropna(subset=['log_return'], inplace=True)

# Target Preservation
y_train_raw = df_train['log_return'].copy()
y_test_raw = df_test['log_return'].copy()

# Outlier Detection
lower_bound = df_train['log_return'].quantile(0.02)
upper_bound = df_train['log_return'].quantile(0.98)

outliers = df_train[(df_train['log_return'] < lower_bound) | (df_train['log_return'] > upper_bound)][['log_return']]
outliers.rename(columns={'log_return': 'raw_log_return'}, inplace=True)

# Apply Clipping
df_train_clipped = df_train.copy()
df_train_clipped['log_return'] = np.clip(df_train['log_return'], lower_bound, upper_bound)
df_test['log_return'] = np.clip(df_test['log_return'], lower_bound, upper_bound)

# Outlier Visualization
plt.figure(figsize=(14, 6), dpi=300)
plt.plot(df_train.index, df_train['log_return'], label='Raw Log Return', color='#FF0000', alpha=0.4)
plt.plot(df_train_clipped.index, df_train_clipped['log_return'], label='Clipped Log Return', color='blue', linewidth=1.5)

plt.title(f"{coin.capitalize()} Train Set: Outlier Clipping Impact on Log Returns")
plt.xlabel("Date")
plt.ylabel("Log Return")
plt.legend(loc='best')
plt.tight_layout()

chart_out = os.path.join(processed_dir, f'{coin}_outlier_log_returns_chart.png')
plt.savefig(chart_out)
plt.close()

df_train = df_train_clipped

# Normalization
scaler = MinMaxScaler(feature_range=(-1, 1))
df_train_scaled = pd.DataFrame(scaler.fit_transform(df_train), index=df_train.index, columns=df_train.columns)
df_test_scaled = pd.DataFrame(scaler.transform(df_test), index=df_test.index, columns=df_test.columns)

scaler_file = os.path.join(processed_dir, f'{coin}_scaler.pkl')
with open(scaler_file, 'wb') as f:
    pickle.dump(scaler, f)

# Wavelet Denoising
def apply_modwt(df):
    df_denoised = df.copy()
    data_length = len(df)
    padded_length = int(np.ceil(data_length / 32.0)) * 32

    for col in df.columns:
        signal = df[col].values
        padded_signal = np.pad(signal, (0, padded_length - data_length), mode='edge')
        coeffs = pywt.swt(padded_signal, 'db2', level=5)

        denoised_coeffs = [(approx, np.zeros_like(detail)) for approx, detail in coeffs]
        denoised_padded = pywt.iswt(denoised_coeffs, 'db2')
        df_denoised[col] = denoised_padded[:data_length]

    return df_denoised

df_train_preprocessed = apply_modwt(df_train_scaled)
df_test_preprocessed = apply_modwt(df_test_scaled)

# Append Raw Targets
df_train_preprocessed['y_raw_log_return'] = y_train_raw
df_test_preprocessed['y_raw_log_return'] = y_test_raw

# Export Final Output Files
train_out = os.path.join(processed_dir, f'{coin}_train_preprocessed.csv')
test_out = os.path.join(processed_dir, f'{coin}_test_preprocessed.csv')
prices_out = os.path.join(processed_dir, f'{coin}_raw_prices.csv')
log_out = os.path.join(processed_dir, f'{coin}_outlier_detected_log.csv')

df_train_preprocessed.to_csv(train_out)
df_test_preprocessed.to_csv(test_out)
df_raw_prices.to_csv(prices_out)
outliers.to_csv(log_out)

print(f"Preprocessed Train shape: {df_train_preprocessed.shape}")
print(f"Preprocessed Test shape: {df_test_preprocessed.shape}")
print(f"Outliers detected and logged: {len(outliers)}")
print(f"Chart saved: {chart_out}")

Train Missing Values (Date, Column):
2025-05-22 - activeaddresses
2025-05-23 - activeaddresses

No missing values in Test Data.
Preprocessed Train shape: (881, 17)
Preprocessed Test shape: (214, 17)
Outliers detected and logged: 36
Chart saved: /content/drive/MyDrive/Crypto Research/DATA/Technical Indicators/ProcessedData/TrainTest/bitcoin_outlier_log_returns_chart.png


In [ ]:
# @title # Features Engineering
import pandas as pd
import numpy as np
import os
import warnings

warnings.filterwarnings('ignore')

# Configuration
coin = 'bitcoin'
base_dir = '/content/drive/MyDrive/Crypto Research/DATA/Technical Indicators'
processed_dir = os.path.join(base_dir, 'ProcessedData/TrainTest')
features_dir = os.path.join(base_dir, 'ProcessedData')
os.makedirs(features_dir, exist_ok=True)

# Load Data
train_file = os.path.join(processed_dir, f'{coin}_train_preprocessed.csv')
test_file = os.path.join(processed_dir, f'{coin}_test_preprocessed.csv')

df_train = pd.read_csv(train_file, index_col='Date', parse_dates=True)
df_test = pd.read_csv(test_file, index_col='Date', parse_dates=True)

# Identify base features for indicator calculation (exclude target-related columns)
exclude_cols = ['y_raw_log_return']
feature_cols = [col for col in df_train.columns if col not in exclude_cols]

# Technical Indicators Calculation
def compute_rsi(series, window):
    if window == 1:
        return pd.Series(50, index=series.index)
    delta = series.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=window).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=window).mean()
    rs = gain / loss
    rsi = 100 - (100 / (1 + rs))
    return rsi.fillna(50)

def generate_technical_indicators(df, base_columns):
    features_dfs = [df.copy()]
    windows = [1, 7, 14]

    for col in base_columns:
        series = df[col]
        df_temp = pd.DataFrame(index=df.index)

        for w in windows:
            if w == 1:
                df_temp[f'{col}_{w}_SMA'] = series
                df_temp[f'{col}_{w}_EMA'] = series
                df_temp[f'{col}_{w}_WMA'] = series
                df_temp[f'{col}_{w}_STD'] = 0.0
                df_temp[f'{col}_{w}_VAR'] = 0.0
                df_temp[f'{col}_{w}_ROC'] = series.pct_change() * 100
                df_temp[f'{col}_{w}_RSI'] = compute_rsi(series, w)
                df_temp[f'{col}_{w}_TRIX'] = series.pct_change() * 100
            else:
                df_temp[f'{col}_{w}_SMA'] = series.rolling(window=w).mean()
                df_temp[f'{col}_{w}_EMA'] = series.ewm(span=w, adjust=False).mean()

                weights = np.arange(1, w + 1)
                df_temp[f'{col}_{w}_WMA'] = series.rolling(window=w).apply(lambda x: np.dot(x, weights) / weights.sum(), raw=True)

                df_temp[f'{col}_{w}_STD'] = series.rolling(window=w).std()
                df_temp[f'{col}_{w}_VAR'] = series.rolling(window=w).var()
                df_temp[f'{col}_{w}_ROC'] = series.pct_change(periods=w) * 100
                df_temp[f'{col}_{w}_RSI'] = compute_rsi(series, w)

                ema1 = series.ewm(span=w, adjust=False).mean()
                ema2 = ema1.ewm(span=w, adjust=False).mean()
                ema3 = ema2.ewm(span=w, adjust=False).mean()
                df_temp[f'{col}_{w}_TRIX'] = ema3.pct_change() * 100

        features_dfs.append(df_temp)

    return pd.concat(features_dfs, axis=1)

# Apply independently to prevent data leakage
df_train_engineered = generate_technical_indicators(df_train, feature_cols)
df_test_engineered = generate_technical_indicators(df_test, feature_cols)

# Replace infinite values generated by pct_change with NaN to be dropped later
df_train_engineered.replace([np.inf, -np.inf], np.nan, inplace=True)
df_test_engineered.replace([np.inf, -np.inf], np.nan, inplace=True)

# Target Formulation (Shift -1)
# Train Target MUST use the raw log return to prevent smoothing leakage
df_train_engineered['Train_Target'] = df_train_engineered['y_raw_log_return'].shift(-1)

# Test Target uses the completely raw log return strictly for evaluation
df_test_engineered['Test_Actual_Target'] = df_test_engineered['y_raw_log_return'].shift(-1)


# Alignment & NaN Handling
# Drop rows with NaNs caused by rolling windows and shifting
df_train_engineered.dropna(inplace=True)
df_test_engineered.dropna(inplace=True)

# Remove the preserved raw logs from the feature sets to prevent leakage
if 'y_raw_log_return' in df_train_engineered.columns:
    df_train_engineered.drop(columns=['y_raw_log_return'], inplace=True)

if 'y_raw_log_return' in df_test_engineered.columns:
    df_test_engineered.drop(columns=['y_raw_log_return'], inplace=True)

# Test Set Isolation
# Extract and isolate the Actual Target from the Test set
y_Test_Actual = df_test_engineered['Test_Actual_Target'].copy()
Test_Features = df_test_engineered.drop(columns=['Test_Actual_Target'])

# Prepare Train set structure
Train_Features = df_train_engineered.copy()

# Generate Feature List for Stage 5
Feature_List = list(Test_Features.columns)

# Expected Outputs Generation
train_features_out = os.path.join(features_dir, f'{coin}_Train_Features.csv')
test_features_out = os.path.join(features_dir, f'{coin}_Test_Features.csv')
test_actual_out = os.path.join(features_dir, f'{coin}_y_Test_Actual.csv')

Train_Features.to_csv(train_features_out)
Test_Features.to_csv(test_features_out)
y_Test_Actual.to_csv(test_actual_out)

print("Complete: Feature Engineering")
print(f"Train_Features shape: {Train_Features.shape} | NaNs: {Train_Features.isna().sum().sum()}")
print(f"Test_Features shape: {Test_Features.shape} | NaNs: {Test_Features.isna().sum().sum()}")
print(f"y_Test_Actual shape: {y_Test_Actual.shape} | NaNs: {y_Test_Actual.isna().sum()}")
print(f"Total Features Generated (Feature_List): {len(Feature_List)}")

Complete: Feature Engineering
Train_Features shape: (866, 401) | NaNs: 0
Test_Features shape: (199, 400) | NaNs: 0
y_Test_Actual shape: (199,) | NaNs: 0
Total Features Generated (Feature_List): 400


In [ ]:
# @title #RANDOMFOREST FEATURES SELECTION
import pandas as pd
import numpy as np
import os
import warnings
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from statsmodels.stats.outliers_influence import variance_inflation_factor

warnings.filterwarnings('ignore')

# Configuration and Paths
coin = 'bitcoin'
base_dir = '/content/drive/MyDrive/Crypto Research/DATA/Technical Indicators'
data_dir = os.path.join(base_dir, 'ProcessedData')

# Define input directory and output directory
features_dir = os.path.join(data_dir)
output_dir = os.path.join(base_dir, 'Features Selection/RandomForestRegressorSelected')
os.makedirs(output_dir, exist_ok=True)

# Load input
train_file = os.path.join(features_dir, f'{coin}_Train_Features.csv')
test_file = os.path.join(features_dir, f'{coin}_Test_Features.csv')

df_train = pd.read_csv(train_file, index_col=0, parse_dates=True)
df_test = pd.read_csv(test_file, index_col=0, parse_dates=True)

# Initialization and Target Separation
y_train = df_train['Train_Target']
X_train = df_train.drop(columns=['Train_Target'])
X_test = df_test.copy()

# Train Random Forest Regressor
print("Training Random Forest Regressor for MDI...")
rf = RandomForestRegressor(n_estimators=100, bootstrap=False, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

# Permutation Importance Filter
print("Calculating Permutation Importance...")
perm_result = permutation_importance(rf, X_train, y_train, n_repeats=10, random_state=42, n_jobs=-1)

importance_df = pd.DataFrame({
    'Feature': X_train.columns,
    'MDI_Score': rf.feature_importances_,
    'Perm_Mean': perm_result.importances_mean,
    'Perm_Std': perm_result.importances_std
})

# Retention Rule
importance_df['Lower_Bound'] = importance_df['Perm_Mean'] - (2 * importance_df['Perm_Std'])
survivors = importance_df[importance_df['Lower_Bound'] > 0].sort_values(by='Perm_Mean', ascending=False)
candidate_features = survivors['Feature'].tolist()

print(f"Features surviving Permutation check: {len(candidate_features)} / {len(Feature_List)}")

# Multicollinearity Check (Pearson Correlation > 0.85)
print("Filtering by Pearson Correlation (> 0.85)...")
X_candidates = X_train[candidate_features]
corr_matrix = X_candidates.corr().abs()
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

to_drop_pearson = set()
for col in upper_tri.columns:
    high_corr_vars = upper_tri.index[upper_tri[col] > 0.85].tolist()
    for var in high_corr_vars:
        score_col = survivors.loc[survivors['Feature'] == col, 'Perm_Mean'].values[0]
        score_var = survivors.loc[survivors['Feature'] == var, 'Perm_Mean'].values[0]

        if score_col < score_var:
            to_drop_pearson.add(col)
        else:
            to_drop_pearson.add(var)

candidate_features = [feat for feat in candidate_features if feat not in to_drop_pearson]
print(f"Features surviving Pearson Correlation check: {len(candidate_features)}")

# Multicollinearity Check (VIF > 10)
print("Filtering by Variance Inflation Factor (VIF > 10)...")
def compute_vif(X, features):
    vif_df = pd.DataFrame()
    vif_df["Feature"] = features
    vif_df["VIF"] = [variance_inflation_factor(X[features].values, i) for i in range(len(features))]
    return vif_df

# Iterative VIF elimination
Best_Features_List = candidate_features.copy()
while len(Best_Features_List) > 0:
    vif_df = compute_vif(X_train, Best_Features_List)
    max_vif = vif_df['VIF'].max()

    if max_vif > 10.0:
        drop_feat = vif_df.loc[vif_df['VIF'].idxmax(), 'Feature']
        Best_Features_List.remove(drop_feat)
    else:
        break

print(f"Final Best Features List Count: {len(Best_Features_List)}")

# Feature Importance Chart Generation (Top 10)
print("Generating Feature Importance Chart (Top 10)...")
plot_df = survivors[survivors['Feature'].isin(Best_Features_List)].sort_values(by='Perm_Mean', ascending=False).head(10)
plot_df = plot_df.sort_values(by='Perm_Mean', ascending=True)

plt.figure(figsize=(10, 6), dpi=300)
plt.barh(plot_df['Feature'], plot_df['Perm_Mean'], color='teal', edgecolor='black')
plt.xlabel('Permutation Importance Score (Mean)')
plt.title(f'{coin.capitalize()} - Top 10 Selected Features')
plt.tight_layout()

chart_file = os.path.join(output_dir, f'{coin}_Feature_Importance_Chart_Top10.png')
plt.savefig(chart_file)
plt.close()

# Feature Pruning, Charting, and Timeframe Export
print("\nGenerating Charts and Exporting selected features grouped by timeframe...")
timeframes = [1, 7, 14]

for tf in timeframes:
    # Filter features specific to this timeframe
    tf_features = [feat for feat in Best_Features_List if f'_{tf}_' in feat]

    if not tf_features:
        print(f"Timeframe {tf}d: 0 features selected. Skipping export.")
        continue

    # Generate Timeframe-Specific Feature Importance Chart (Top 10)
    plot_df_tf = survivors[survivors['Feature'].isin(tf_features)].sort_values(by='Perm_Mean', ascending=False).head(10)
    plot_df_tf = plot_df_tf.sort_values(by='Perm_Mean', ascending=True)

    plt.figure(figsize=(10, 6), dpi=300)
    plt.barh(plot_df_tf['Feature'], plot_df_tf['Perm_Mean'], color='teal', edgecolor='black')
    plt.xlabel('Permutation Importance Score (Mean)')
    plt.title(f'{coin.capitalize()} - Top 10 Selected Features ({tf}-Day Timeframe)')
    plt.tight_layout()

    chart_tf_file = os.path.join(output_dir, f'{coin}_Feature_Importance_Chart_tf{tf}.png')
    plt.savefig(chart_tf_file)
    plt.close()

    # Prepare final DataFrames for the specific timeframe
    df_train_tf = df_train[tf_features + ['Train_Target']].copy()
    df_test_tf = df_test[tf_features].copy()

    # Define file paths
    train_tf_file = os.path.join(output_dir, f'{coin}_Train_Final_tf{tf}.csv')
    test_tf_file = os.path.join(output_dir, f'{coin}_Test_Final_tf{tf}.csv')

    # Export to CSV
    df_train_tf.to_csv(train_tf_file)
    df_test_tf.to_csv(test_tf_file)

    print(f"Timeframe {tf}d: Exported {len(tf_features)} features.")
    print(f" Chart: {chart_tf_file}")
    print(f" Train: {train_tf_file}")
    print(f" Test:  {test_tf_file}")


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Crypto Research/DATA/Technical Indicators/ProcessedData/bitcoin_Train_Features.csv'

In [ ]:
# @title #BORUTA FEATURES SELECTION USING
# Install required package
!pip install boruta

import pandas as pd
import numpy as np
import os
import warnings
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from boruta import BorutaPy
from statsmodels.stats.outliers_influence import variance_inflation_factor

warnings.filterwarnings('ignore')

# Configuration and paths
coin = 'bitcoin'
base_dir = '/content/drive/MyDrive/Crypto Research/DATA/Technical Indicators'
processed_dir = os.path.join(base_dir, 'ProcessedData/TrainTest')
features_dir = os.path.join(base_dir, 'ProcessedData')
output_dir = os.path.join(base_dir, 'Features Selection/BorutaSelected')

os.makedirs(features_dir, exist_ok=True)
os.makedirs(output_dir, exist_ok=True)

# Path definitions
train_prep_file = os.path.join(processed_dir, f'{coin}_train_preprocessed.csv')
test_prep_file = os.path.join(processed_dir, f'{coin}_test_preprocessed.csv')
train_features_file = os.path.join(features_dir, f'{coin}_Train_Features.csv')
test_features_file = os.path.join(features_dir, f'{coin}_Test_Features.csv')

# Data loading
if not all(os.path.exists(f) for f in [train_prep_file, test_prep_file, train_features_file, test_features_file]):
    print("ERROR: Required input files not found.")
else:
    df_train_prep = pd.read_csv(train_prep_file, index_col='Date', parse_dates=True)
    df_test_prep = pd.read_csv(test_prep_file, index_col='Date', parse_dates=True)
    df_train_feat = pd.read_csv(train_features_file, index_col='Date', parse_dates=True)
    df_test_feat = pd.read_csv(test_features_file, index_col='Date', parse_dates=True)

    # Target generation
    timeframes = [1, 7, 14]
    target_cols = []

    for tf in timeframes:
        col_name = f'target_{tf}d'
        target_cols.append(col_name)
        df_train_feat[col_name] = df_train_prep['y_raw_log_return'].shift(-1).rolling(window=tf).sum().shift(-(tf-1))
        df_test_feat[col_name] = df_test_prep['y_raw_log_return'].shift(-1).rolling(window=tf).sum().shift(-(tf-1))

    # Export test actuals
    test_actual_out = os.path.join(features_dir, f'{coin}_y_Test_Actual.csv')
    df_test_feat[target_cols].to_csv(test_actual_out)
    print(f"Exported Actual Test Targets to: {test_actual_out}")

    # Variance inflation factor calculation
    def calculate_vif(X, threshold=10.0, min_features=2):
        if X is None or X.shape[1] < min_features:
            return list(X.columns) if X is not None else []

        Xc = X.copy()
        nunique = Xc.nunique(dropna=False)
        Xc = Xc.loc[:, nunique > 1]
        Xc = Xc.replace([np.inf, -np.inf], np.nan).dropna(axis=0, how='any')

        if Xc.shape[1] < min_features:
            return list(Xc.columns)

        features = Xc.columns.tolist()
        while True:
            if len(features) < min_features: break
            vif_vals = []
            for i in range(len(features)):
                try:
                    vif_vals.append(variance_inflation_factor(Xc[features].values, i))
                except:
                    vif_vals.append(np.inf)

            vif_data = pd.DataFrame({'feature': features, 'VIF': vif_vals})
            if vif_data.empty: break

            max_vif = float(vif_data['VIF'].max())
            if not np.isfinite(max_vif) or max_vif > threshold:
                drop_feature = vif_data.loc[vif_data['VIF'].idxmax(), 'feature']
                features.remove(drop_feature)
            else:
                break
        return features

    # Feature selection pipeline
    for tf in timeframes:
        print(f"\nProcessing Timeframe {tf}d")
        target_col = f'target_{tf}d'

        # Format and clean dataframe
        df_train_tf = df_train_feat.copy()
        df_train_tf = df_train_tf.replace([np.inf, -np.inf], np.nan)

        # Strict data leakage prevention
        leakage_cols = [col for col in df_train_tf.columns if 'target' in str(col).lower() or 'return' in str(col).lower() or 'Train_Target' in str(col)]
        leakage_cols = [col for col in leakage_cols if col != target_col]
        df_train_tf = df_train_tf.drop(columns=leakage_cols, errors='ignore')

        # Drop any remaining NaNs across target and features
        df_train_tf = df_train_tf.dropna()

        if df_train_tf.empty:
            print(f"Warning: Dataset is empty after dropping NaNs for {tf}d. Skipping.")
            continue

        # Isolate features and target
        X_train = df_train_tf.drop(columns=[target_col])
        y_train = df_train_tf[target_col]

        # Boruta algorithm
        print("Starting Boruta Algorithm")
        rf_boruta = RandomForestRegressor(n_jobs=-1, max_depth=5, random_state=42)
        boruta_selector = BorutaPy(rf_boruta, n_estimators='auto', verbose=0, random_state=42, max_iter=100)
        boruta_selector.fit(X_train.values, y_train.values)

        boruta_features = X_train.columns[boruta_selector.support_].tolist()
        print(f"Boruta Confirmed Features: {len(boruta_features)}")

        if not boruta_features:
            continue

        # MDI scores calculation
        print("Calculating MDI Scores")
        rf_mdi = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
        rf_mdi.fit(X_train[boruta_features], y_train)
        mdi_scores = dict(zip(boruta_features, rf_mdi.feature_importances_))

        # Pearson filter
        print("Applying Pearson Filter")
        X_boruta = X_train[boruta_features]
        corr_matrix = X_boruta.corr().abs()
        upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

        to_drop_pearson = set()
        for col in upper_tri.columns:
            high_corr_vars = upper_tri.index[upper_tri[col] > 0.85].tolist()
            for var in high_corr_vars:
                if mdi_scores[col] < mdi_scores[var]:
                    to_drop_pearson.add(col)
                else:
                    to_drop_pearson.add(var)

        features_after_corr = [feat for feat in boruta_features if feat not in to_drop_pearson]
        print(f"Features after Pearson: {len(features_after_corr)}")

        # VIF filter
        print("Applying VIF Filter")
        X_vif = X_train[features_after_corr]
        final_features = calculate_vif(X_vif, threshold=10.0)
        print(f"Features after VIF: {len(final_features)}")

        # Visualization and export
        final_scores = {feat: mdi_scores[feat] for feat in final_features}
        if final_scores:
            plot_df = pd.DataFrame(list(final_scores.items()), columns=['Feature', 'MDI_Score'])
            plot_df = plot_df.sort_values(by='MDI_Score', ascending=False).head(10).sort_values(by='MDI_Score', ascending=True)

            plt.figure(figsize=(10, 6), dpi=300)
            plt.barh(plot_df['Feature'], plot_df['MDI_Score'], color='teal', edgecolor='black')
            plt.xlabel('MDI Importance Score')
            plt.title(f'{coin.capitalize()} - Top 10 Boruta Features ({tf}d Timeframe)')
            plt.tight_layout()

            chart_file = os.path.join(output_dir, f'{coin}_Boruta_Top10_Chart_{tf}d.png')
            plt.savefig(chart_file)
            plt.close()

        # Final validation and saving
        final_features_valid = [f for f in final_features if f in df_train_tf.columns and f in df_test_feat.columns]
        if not final_features_valid:
            print(f"Warning: No valid features found for {tf}d timeframe. Skipping.")
            continue

        final_columns = final_features_valid + [target_col]

        # Test dataset formatting
        df_test_final = df_test_feat.copy()

        # Prepare final train/test sets
        df_train_final = df_train_tf[final_columns]
        if target_col in df_test_final.columns:
            df_test_final = df_test_final[final_columns]
        else:
            df_test_final = df_test_final[final_features_valid]

        # Export final data
        train_output_path = os.path.join(output_dir, f'{coin}_train_boruta_{tf}d.csv')
        test_output_path = os.path.join(output_dir, f'{coin}_test_boruta_{tf}d.csv')

        df_train_final.to_csv(train_output_path)
        df_test_final.to_csv(test_output_path)
        print(f"Exported Boruta-selected features for {tf}d timeframe.")

Exported Actual Test Targets to: /content/drive/MyDrive/Crypto Research/DATA/Technical Indicators/ProcessedData/bitcoin_y_Test_Actual.csv

Processing Timeframe 1d
Starting Boruta Algorithm
Boruta Confirmed Features: 2
Calculating MDI Scores
Applying Pearson Filter
Features after Pearson: 2
Applying VIF Filter
Features after VIF: 2
Exported Boruta-selected features for 1d timeframe.

Processing Timeframe 7d
Starting Boruta Algorithm
Boruta Confirmed Features: 98
Calculating MDI Scores
Applying Pearson Filter
Features after Pearson: 55
Applying VIF Filter
Features after VIF: 49
Exported Boruta-selected features for 7d timeframe.

Processing Timeframe 14d
Starting Boruta Algorithm
Boruta Confirmed Features: 184
Calculating MDI Scores
Applying Pearson Filter
Features after Pearson: 65
Applying VIF Filter
Features after VIF: 57
Exported Boruta-selected features for 14d timeframe.


In [ ]:
# @title #ARIMA BASELINE MODEL
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings

warnings.filterwarnings('ignore')

# Configuration and Paths
coin = 'bitcoin'
base_dir = '/content/drive/MyDrive/Crypto Research/DATA/Technical Indicators'
input_dir = os.path.join(base_dir, 'ProcessedData/TrainTest')
output_dir = '/content/drive/MyDrive/Crypto Research/RESULTS/ARIMA'
os.makedirs(output_dir, exist_ok=True)

# Load Preprocessed Data
train_file = os.path.join(input_dir, f'{coin}_train_preprocessed.csv')
test_file = os.path.join(input_dir, f'{coin}_test_preprocessed.csv')

df_train_prep = pd.read_csv(train_file, index_col='Date', parse_dates=True)
df_test_prep = pd.read_csv(test_file, index_col='Date', parse_dates=True)

timeframes = [1, 7, 14]

# Outlier Clipping for Training Stability
lower_bound = df_train_prep['y_raw_log_return'].quantile(0.02)
upper_bound = df_train_prep['y_raw_log_return'].quantile(0.98)

df_train_prep['y_clipped_log_return'] = np.clip(df_train_prep['y_raw_log_return'], lower_bound, upper_bound)
# Note: We do NOT clip the test set because we want to evaluate on reality

timeframes = [1, 7, 14]

for tf in timeframes:
    print(f"\nProcessing Auto ARIMA Baseline for Timeframe: {tf}d")

    # 1. GENERATE TRAIN TARGET (Using CLIPPED data to find stable patterns)
    y_train_tf = df_train_prep['y_clipped_log_return'].shift(-1).rolling(window=tf).sum().shift(-(tf-1))
    y_train = y_train_tf.replace([np.inf, -np.inf], np.nan).dropna()

    # 2. GENERATE TEST TARGET (Using RAW data to evaluate real-world performance)
    y_test_tf = df_test_prep['y_raw_log_return'].shift(-1).rolling(window=tf).sum().shift(-(tf-1))
    y_test = y_test_tf.replace([np.inf, -np.inf], np.nan).dropna()

    if y_train.empty or y_test.empty:
        print(f"Warning: Cleaned target data is empty for {tf}d. Skipping.")
        continue

    # Train Auto ARIMA model on Clipped Target
    print(f"Finding optimal ARIMA parameters for {tf}d on Clipped Data...")
    try:
        model_fit = auto_arima(
            y_train,
            start_p=0, start_q=0,
            max_p=5, max_q=5,
            d=0,
            seasonal=False,
            trace=False,
            error_action='ignore',
            suppress_warnings=True,
            stepwise=True
        )
        print(f"Optimal Model Selected: {model_fit.order}")
    except Exception as e:
        print(f"Error fitting Auto ARIMA for {tf}d: {e}")
        continue

    # Forecasting
    print("Generating predictions...")
    predictions = model_fit.predict(n_periods=len(y_test))
    predictions.index = y_test.index

    # Evaluation Metrics (Scoring against RAW Target)
    print("Evaluating predictions against RAW Real-world Data...")
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    mae = mean_absolute_error(y_test, predictions)
    print(f"Evaluation Metrics [{tf}d]:")
    print(f"RMSE: {rmse:.6f}")
    print(f"MAE:  {mae:.6f}")

    # Export Predictions
    results_df = pd.DataFrame({
        'Actual_Raw_Target': y_test,
        'Predicted_Target': predictions
    })
    results_path = os.path.join(output_dir, f'{coin}_AutoARIMA_predictions_{tf}d.csv')
    results_df.to_csv(results_path)

    # Plotting Results
    plt.figure(figsize=(14, 7), dpi=300)

    # Plot Train (Clipped) and Test (Raw)
    plt.plot(y_train.index, y_train, label='Train Target (Clipped)', color='blue', alpha=0.3)
    plt.plot(y_test.index, y_test, label='Test Actual (Raw/Unclipped)', color='black', linewidth=1.5)

    optimal_order = model_fit.order
    plt.plot(predictions.index, predictions, label=f'ARIMA{optimal_order} Forecast', color='red', linestyle='--', linewidth=2)

    plt.title(f"{coin.capitalize()} Real-World Target Prediction using Auto ARIMA ({tf}d Timeframe)")
    plt.xlabel("Date")
    plt.ylabel(f"Log Return ({tf}d rolling sum)")
    plt.legend(loc='best')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    chart_path = os.path.join(output_dir, f'{coin}_AutoARIMA_forecast_{tf}d.png')
    plt.savefig(chart_path)
    plt.close()

    print(f"Chart saved to: {chart_path}")

print("\nMulti-timeframe Auto ARIMA Baseline generation complete.")


Processing Auto ARIMA Baseline for Timeframe: 1d
Finding optimal ARIMA parameters for 1d on Clipped Data...
Optimal Model Selected: (1, 0, 0)
Generating predictions...
Evaluating predictions against RAW Real-world Data...
Evaluation Metrics [1d]:
RMSE: 0.015917
MAE:  0.011234
Chart saved to: /content/drive/MyDrive/Crypto Research/RESULTS/ARIMA/bitcoin_AutoARIMA_forecast_chart_full_1d.png

Processing Auto ARIMA Baseline for Timeframe: 7d
Finding optimal ARIMA parameters for 7d on Clipped Data...
Optimal Model Selected: (5, 0, 4)
Generating predictions...
Evaluating predictions against RAW Real-world Data...
Evaluation Metrics [7d]:
RMSE: 0.050088
MAE:  0.038398
Chart saved to: /content/drive/MyDrive/Crypto Research/RESULTS/ARIMA/bitcoin_AutoARIMA_forecast_chart_full_7d.png

Processing Auto ARIMA Baseline for Timeframe: 14d
Finding optimal ARIMA parameters for 14d on Clipped Data...
Optimal Model Selected: (2, 0, 0)
Generating predictions...
Evaluating predictions against RAW Real-world

In [ ]:
# @title LSTM MODEL WITH BORUTA
# !pip install tensorflow scikit-learn

import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import warnings

warnings.filterwarnings('ignore')

# Configuration and paths
coin = 'bitcoin'
base_dir = '/content/drive/MyDrive/Crypto Research/DATA/Technical Indicators'
prep_dir = os.path.join(base_dir, 'ProcessedData/TrainTest')
features_dir = os.path.join(base_dir, 'Features Selection/BorutaSelected')
output_dir = os.path.join(base_dir, 'RESULTS/LSTM_Boruta')
os.makedirs(output_dir, exist_ok=True)

# Sequence length for LSTM lookback
TIME_STEPS = 14

# Load preprocessed data for targets
train_prep_file = os.path.join(prep_dir, f'{coin}_train_preprocessed.csv')
test_prep_file = os.path.join(prep_dir, f'{coin}_test_preprocessed.csv')

df_train_prep = pd.read_csv(train_prep_file, index_col='Date', parse_dates=True)
df_test_prep = pd.read_csv(test_prep_file, index_col='Date', parse_dates=True)

# Outlier clipping for training target stability
lower_bound = df_train_prep['y_raw_log_return'].quantile(0.02)
upper_bound = df_train_prep['y_raw_log_return'].quantile(0.98)
df_train_prep['y_clipped_log_return'] = np.clip(df_train_prep['y_raw_log_return'], lower_bound, upper_bound)

# Sequence generation function for 3D LSTM input
def create_sequences(X, y, time_steps):
    Xs, ys, indices = [], [], []
    for i in range(len(X) - time_steps):
        Xs.append(X.iloc[i:(i + time_steps)].values)
        ys.append(y.iloc[i + time_steps])
        indices.append(y.index[i + time_steps])
    return np.array(Xs), np.array(ys), indices

timeframes = [1, 7, 14]

# LSTM pipeline for each timeframe
for tf in timeframes:
    print(f"\nProcessing LSTM for Timeframe: {tf}d")

    # Load Boruta selected features
    train_feat_file = os.path.join(features_dir, f'{coin}_train_boruta_{tf}d.csv')
    test_feat_file = os.path.join(features_dir, f'{coin}_test_boruta_{tf}d.csv')

    if not os.path.exists(train_feat_file) or not os.path.exists(test_feat_file):
        print(f"Warning: Feature files for {tf}d not found. Skipping.")
        continue

    df_train_feat = pd.read_csv(train_feat_file, index_col='Date', parse_dates=True)
    df_test_feat = pd.read_csv(test_feat_file, index_col='Date', parse_dates=True)

    # Generate rolling targets
    y_train_tf = df_train_prep['y_clipped_log_return'].shift(-1).rolling(window=tf).sum().shift(-(tf-1))
    y_test_tf = df_test_prep['y_raw_log_return'].shift(-1).rolling(window=tf).sum().shift(-(tf-1))

    # Align features and targets
    target_col_name = f'target_{tf}d'
    train_aligned = pd.concat([df_train_feat.drop(columns=[c for c in df_train_feat.columns if 'target' in c.lower()], errors='ignore'), y_train_tf.rename(target_col_name)], axis=1).dropna()
    test_aligned = pd.concat([df_test_feat.drop(columns=[c for c in df_test_feat.columns if 'target' in c.lower()], errors='ignore'), y_test_tf.rename(target_col_name)], axis=1).dropna()

    X_train_raw = train_aligned.drop(columns=[target_col_name])
    y_train_raw = train_aligned[target_col_name]
    X_test_raw = test_aligned.drop(columns=[target_col_name])
    y_test_raw = test_aligned[target_col_name]

    # Scale features and targets
    scaler_X = StandardScaler()
    scaler_y = StandardScaler()

    X_train_scaled = pd.DataFrame(scaler_X.fit_transform(X_train_raw), columns=X_train_raw.columns, index=X_train_raw.index)
    X_test_scaled = pd.DataFrame(scaler_X.transform(X_test_raw), columns=X_test_raw.columns, index=X_test_raw.index)

    y_train_scaled = pd.Series(scaler_y.fit_transform(y_train_raw.values.reshape(-1, 1)).flatten(), index=y_train_raw.index)
    y_test_scaled = pd.Series(scaler_y.transform(y_test_raw.values.reshape(-1, 1)).flatten(), index=y_test_raw.index)

    # Create 3D sequences
    X_train_seq, y_train_seq, train_idx = create_sequences(X_train_scaled, y_train_scaled, TIME_STEPS)
    X_test_seq, y_test_seq, test_idx = create_sequences(X_test_scaled, y_test_scaled, TIME_STEPS)

    if len(X_train_seq) == 0 or len(X_test_seq) == 0:
        print("Error: Not enough data points to create sequences. Reduce TIME_STEPS.")
        continue

    # Build LSTM architecture
    print("Building and Training LSTM Model...")
    model = Sequential()
    model.add(LSTM(units=50, return_sequences=False, input_shape=(X_train_seq.shape[1], X_train_seq.shape[2])))
    model.add(Dropout(0.2))
    model.add(Dense(units=1))

    model.compile(optimizer='adam', loss='mean_squared_error')

    # Early stopping configuration
    early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

    # Train model
    history = model.fit(
        X_train_seq, y_train_seq,
        epochs=100,
        batch_size=32,
        validation_split=0.1,
        callbacks=[early_stop],
        verbose=0
    )

    # Forecasting and inverse transformation
    print("Generating predictions...")
    predictions_scaled = model.predict(X_test_seq, verbose=0)
    predictions = scaler_y.inverse_transform(predictions_scaled).flatten()
    pred_series = pd.Series(predictions, index=test_idx)

    # Retrieve actual raw targets for evaluation
    actual_raw = y_test_raw.loc[test_idx]

    # Evaluation metrics including Directional Accuracy
    rmse = np.sqrt(mean_squared_error(actual_raw, pred_series))
    mae = mean_absolute_error(actual_raw, pred_series)

    actual_direction = np.sign(actual_raw)
    pred_direction = np.sign(pred_series)
    directional_accuracy = np.mean(actual_direction == pred_direction) * 100

    print(f"Evaluation Metrics [{tf}d]:")
    print(f"RMSE: {rmse:.6f}")
    print(f"MAE:  {mae:.6f}")
    print(f"Directional Accuracy: {directional_accuracy:.2f}%\n")

    # Export predictions
    results_df = pd.DataFrame({
        'Actual_Raw_Target': actual_raw,
        'Predicted_Target': pred_series
    })
    results_path = os.path.join(output_dir, f'{coin}_LSTM_predictions_{tf}d.csv')
    results_df.to_csv(results_path)

    # Plotting results
    plt.figure(figsize=(14, 7), dpi=300)
    plt.plot(actual_raw.index, actual_raw, label='Test Actual (Raw)', color='black', linewidth=1.5)
    plt.plot(pred_series.index, pred_series, label='LSTM Forecast', color='red', linestyle='--', linewidth=2)

    plt.title(f"{coin.capitalize()} LSTM Prediction ({tf}d Timeframe) using RF Features")
    plt.xlabel("Date")
    plt.ylabel(f"Log Return ({tf}d rolling sum)")
    plt.legend(loc='best')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    chart_path = os.path.join(output_dir, f'{coin}_LSTM_forecast_chart_{tf}d.png')
    plt.savefig(chart_path)
    plt.close()

    print(f"Chart saved to: {chart_path}")

print("\nMulti-timeframe LSTM generation complete.")


Processing LSTM for Timeframe: 1d
Building and Training LSTM Model...
Generating predictions...
Evaluation Metrics [1d]:
RMSE: 0.016302
MAE:  0.011435
Directional Accuracy: 55.14%

Chart saved to: /content/drive/MyDrive/Crypto Research/DATA/Technical Indicators/RESULTS/LSTM_Boruta/bitcoin_LSTM_forecast_chart_1d.png

Processing LSTM for Timeframe: 7d
Building and Training LSTM Model...
Generating predictions...
Evaluation Metrics [7d]:
RMSE: 0.053633
MAE:  0.042775
Directional Accuracy: 53.07%

Chart saved to: /content/drive/MyDrive/Crypto Research/DATA/Technical Indicators/RESULTS/LSTM_Boruta/bitcoin_LSTM_forecast_chart_7d.png

Processing LSTM for Timeframe: 14d
Building and Training LSTM Model...
Generating predictions...
Evaluation Metrics [14d]:
RMSE: 0.058418
MAE:  0.047306
Directional Accuracy: 56.98%

Chart saved to: /content/drive/MyDrive/Crypto Research/DATA/Technical Indicators/RESULTS/LSTM_Boruta/bitcoin_LSTM_forecast_chart_14d.png

Multi-timeframe LSTM generation complete.


In [ ]:
# @title LSTM MODEL WITH RANDOM FOREST REGRESSOR FEATURES
# !pip install tensorflow scikit-learn

import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import warnings

warnings.filterwarnings('ignore')

# Configuration and paths
coin = 'bitcoin'
base_dir = '/content/drive/MyDrive/Crypto Research/DATA/Technical Indicators'
prep_dir = os.path.join(base_dir, 'ProcessedData/TrainTest')

# Pointing to the Random Forest feature selection directory
features_dir = os.path.join(base_dir, 'Features Selection/RandomForestRegressorSelected')
output_dir = os.path.join(base_dir, 'RESULTS/LSTM_RandomForestFeatures')
os.makedirs(output_dir, exist_ok=True)

# Sequence length for LSTM lookback
TIME_STEPS = 14

# Load preprocessed data for targets
train_prep_file = os.path.join(prep_dir, f'{coin}_train_preprocessed.csv')
test_prep_file = os.path.join(prep_dir, f'{coin}_test_preprocessed.csv')

df_train_prep = pd.read_csv(train_prep_file, index_col='Date', parse_dates=True)
df_test_prep = pd.read_csv(test_prep_file, index_col='Date', parse_dates=True)

# Outlier clipping for training target stability
lower_bound = df_train_prep['y_raw_log_return'].quantile(0.02)
upper_bound = df_train_prep['y_raw_log_return'].quantile(0.98)
df_train_prep['y_clipped_log_return'] = np.clip(df_train_prep['y_raw_log_return'], lower_bound, upper_bound)

# Sequence generation function for 3D LSTM input
def create_sequences(X, y, time_steps):
    Xs, ys, indices = [], [], []
    for i in range(len(X) - time_steps):
        Xs.append(X.iloc[i:(i + time_steps)].values)
        ys.append(y.iloc[i + time_steps])
        indices.append(y.index[i + time_steps])
    return np.array(Xs), np.array(ys), indices

timeframes = [1, 7, 14]

# LSTM pipeline for each timeframe
for tf in timeframes:
    print(f"\nProcessing LSTM for Timeframe: {tf}d using Random Forest Features")

    # Load Random Forest selected features (Matching your Phase 1 naming convention)
    train_feat_file = os.path.join(features_dir, f'{coin}_Train_Final_tf{tf}.csv')
    test_feat_file = os.path.join(features_dir, f'{coin}_Test_Final_tf{tf}.csv')

    if not os.path.exists(train_feat_file) or not os.path.exists(test_feat_file):
        print(f"Warning: Feature files for {tf}d not found in {features_dir}. Skipping.")
        continue

    df_train_feat = pd.read_csv(train_feat_file, index_col='Date', parse_dates=True)
    df_test_feat = pd.read_csv(test_feat_file, index_col='Date', parse_dates=True)

    # Generate rolling targets
    y_train_tf = df_train_prep['y_clipped_log_return'].shift(-1).rolling(window=tf).sum().shift(-(tf-1))
    y_test_tf = df_test_prep['y_raw_log_return'].shift(-1).rolling(window=tf).sum().shift(-(tf-1))

    # Align features and targets
    target_col_name = f'target_{tf}d'

    # Drop any legacy target columns that might have been saved in the feature files to prevent leakage
    drop_cols_train = [c for c in df_train_feat.columns if 'target' in c.lower()]
    drop_cols_test = [c for c in df_test_feat.columns if 'target' in c.lower()]

    train_aligned = pd.concat([df_train_feat.drop(columns=drop_cols_train, errors='ignore'), y_train_tf.rename(target_col_name)], axis=1).dropna()
    test_aligned = pd.concat([df_test_feat.drop(columns=drop_cols_test, errors='ignore'), y_test_tf.rename(target_col_name)], axis=1).dropna()

    X_train_raw = train_aligned.drop(columns=[target_col_name])
    y_train_raw = train_aligned[target_col_name]
    X_test_raw = test_aligned.drop(columns=[target_col_name])
    y_test_raw = test_aligned[target_col_name]

    # Scale features and targets
    scaler_X = StandardScaler()
    scaler_y = StandardScaler()

    X_train_scaled = pd.DataFrame(scaler_X.fit_transform(X_train_raw), columns=X_train_raw.columns, index=X_train_raw.index)
    X_test_scaled = pd.DataFrame(scaler_X.transform(X_test_raw), columns=X_test_raw.columns, index=X_test_raw.index)

    y_train_scaled = pd.Series(scaler_y.fit_transform(y_train_raw.values.reshape(-1, 1)).flatten(), index=y_train_raw.index)
    y_test_scaled = pd.Series(scaler_y.transform(y_test_raw.values.reshape(-1, 1)).flatten(), index=y_test_raw.index)

    # Create 3D sequences
    X_train_seq, y_train_seq, train_idx = create_sequences(X_train_scaled, y_train_scaled, TIME_STEPS)
    X_test_seq, y_test_seq, test_idx = create_sequences(X_test_scaled, y_test_scaled, TIME_STEPS)

    if len(X_train_seq) == 0 or len(X_test_seq) == 0:
        print("Error: Not enough data points to create sequences. Reduce TIME_STEPS.")
        continue

    # Build LSTM architecture
    print("Building and Training LSTM Model...")
    model = Sequential()
    model.add(LSTM(units=50, return_sequences=False, input_shape=(X_train_seq.shape[1], X_train_seq.shape[2])))
    model.add(Dropout(0.2))
    model.add(Dense(units=1))

    model.compile(optimizer='adam', loss='mean_squared_error')

    # Early stopping configuration
    early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

    # Train model
    history = model.fit(
        X_train_seq, y_train_seq,
        epochs=100,
        batch_size=32,
        validation_split=0.1,
        callbacks=[early_stop],
        verbose=0
    )

    # Forecasting and inverse transformation
    print("Generating predictions...")
    predictions_scaled = model.predict(X_test_seq, verbose=0)
    predictions = scaler_y.inverse_transform(predictions_scaled).flatten()
    pred_series = pd.Series(predictions, index=test_idx)

    # Retrieve actual raw targets for evaluation
    actual_raw = y_test_raw.loc[test_idx]

    # Evaluation metrics including Directional Accuracy
    rmse = np.sqrt(mean_squared_error(actual_raw, pred_series))
    mae = mean_absolute_error(actual_raw, pred_series)

    actual_direction = np.sign(actual_raw)
    pred_direction = np.sign(pred_series)
    directional_accuracy = np.mean(actual_direction == pred_direction) * 100

    print(f"Evaluation Metrics [{tf}d]:")
    print(f"RMSE: {rmse:.6f}")
    print(f"MAE:  {mae:.6f}")
    print(f"Directional Accuracy: {directional_accuracy:.2f}%\n")

    # Export predictions
    results_df = pd.DataFrame({
        'Actual_Raw_Target': actual_raw,
        'Predicted_Target': pred_series
    })
    results_path = os.path.join(output_dir, f'{coin}_LSTM_RF_predictions_{tf}d.csv')
    results_df.to_csv(results_path)

    # Plotting results
    plt.figure(figsize=(14, 7), dpi=300)
    plt.plot(actual_raw.index, actual_raw, label='Test Actual (Raw)', color='black', linewidth=1.5)
    plt.plot(pred_series.index, pred_series, label='LSTM Forecast', color='red', linestyle='--', linewidth=2)

    plt.title(f"{coin.capitalize()} LSTM Prediction ({tf}d Timeframe) using Random Forest Features")
    plt.xlabel("Date")
    plt.ylabel(f"Log Return ({tf}d rolling sum)")
    plt.legend(loc='best')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    chart_path = os.path.join(output_dir, f'{coin}_LSTM_RF_forecast_chart_{tf}d.png')
    plt.savefig(chart_path)
    plt.close()

    print(f"Chart saved to: {chart_path}")

print("\nMulti-timeframe LSTM generation complete.")


Processing LSTM for Timeframe: 1d using Random Forest Features
Building and Training LSTM Model...
Generating predictions...
Evaluation Metrics [1d]:
RMSE: 0.016296
MAE:  0.011432
Directional Accuracy: 54.59%

Chart saved to: /content/drive/MyDrive/Crypto Research/DATA/Technical Indicators/RESULTS/LSTM_RandomForestFeatures/bitcoin_LSTM_RF_forecast_chart_1d.png

Processing LSTM for Timeframe: 7d using Random Forest Features
Building and Training LSTM Model...
Generating predictions...
Evaluation Metrics [7d]:
RMSE: 0.090959
MAE:  0.070250
Directional Accuracy: 49.72%

Chart saved to: /content/drive/MyDrive/Crypto Research/DATA/Technical Indicators/RESULTS/LSTM_RandomForestFeatures/bitcoin_LSTM_RF_forecast_chart_7d.png

Processing LSTM for Timeframe: 14d using Random Forest Features
Building and Training LSTM Model...
Generating predictions...
Evaluation Metrics [14d]:
RMSE: 0.102921
MAE:  0.081074
Directional Accuracy: 42.44%

Chart saved to: /content/drive/MyDrive/Crypto Research/DATA